## Libs

In [2]:
import random
import datetime
import numpy as np

import scipy.sparse as sp
import pandas as pd

from itertools import islice, cycle
from more_itertools import pairwise
from implicit.nearest_neighbours import TFIDFRecommender

%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

import seaborn as sns
sns.set(style='whitegrid')
sns.set(rc={'figure.figsize':(17, 9)})

from IPython.display import display, HTML, clear_output
display(HTML('<style>.container { width:80% !important; }</style>'))
display(HTML('<style>.prompt { min-width:10ex !important; }</style>'))
display(HTML('<style>div#notebook { font-size:12px !important; }</style>'))

## Functions

In [3]:
def calculate_novelty(train_interactions, recommendations, top_n): 
    users = recommendations['user_id'].unique()
    n_users = train_interactions['user_id'].nunique()
    n_users_per_item = train_interactions.groupby('item_id')['user_id'].nunique()

    recommendations = recommendations.loc[recommendations['rank'] <= top_n].copy()
    recommendations['n_users_per_item'] = recommendations['item_id'].map(n_users_per_item)
    recommendations['n_users_per_item'] = recommendations['n_users_per_item'].fillna(1)
    recommendations['item_novelty'] = -np.log2(recommendations['n_users_per_item'] / n_users)

    item_novelties = recommendations[['user_id', 'rank', 'item_novelty']]
    
    miuf_at_k = item_novelties.loc[item_novelties['rank'] <= top_n, ['user_id', 'item_novelty']]
    miuf_at_k = miuf_at_k.groupby('user_id').agg('mean').squeeze()

    return miuf_at_k.reindex(users).mean()

In [4]:
def compute_metrics(train, test, recs, top_N):
    result = {}
    test_recs = test.set_index(['user_id', 'item_id']).join(recs.set_index(['user_id', 'item_id']))
    test_recs = test_recs.sort_values(by=['user_id', 'rank'])

    test_recs['users_item_count'] = test_recs.groupby(level='user_id')['rank'].transform(np.size)
    test_recs['reciprocal_rank'] = (1 / test_recs['rank']).fillna(0)
    test_recs['cumulative_rank'] = test_recs.groupby(level='user_id').cumcount() + 1
    test_recs['cumulative_rank'] = test_recs['cumulative_rank'] / test_recs['rank']
    
    users_count = test_recs.index.get_level_values('user_id').nunique()
    
    # Uncomment for Precision/Recall at k results

#     for k in range(1, top_N + 1):
#         hit_k = f'hit@{k}'
#         test_recs[hit_k] = test_recs['rank'] <= k
#         result[f'Precision@{k}'] = (test_recs[hit_k] / k).sum() / users_count
#         result[f'Recall@{k}'] = (test_recs[hit_k] / test_recs['users_item_count']).sum() / users_count
        
    result[f'MAP@{top_N}'] = (test_recs['cumulative_rank'] / test_recs['users_item_count']).sum() / users_count
    result[f'Novelty@{top_N}'] = calculate_novelty(train, recs, top_N)
    
    return pd.Series(result)

In [5]:
ranges = pd.date_range(
    start="2025-01-01",
    end="2026-01-01",
    freq="D"
)
ranges

DatetimeIndex(['2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04',
               '2025-01-05', '2025-01-06', '2025-01-07', '2025-01-08',
               '2025-01-09', '2025-01-10',
               ...
               '2025-12-23', '2025-12-24', '2025-12-25', '2025-12-26',
               '2025-12-27', '2025-12-28', '2025-12-29', '2025-12-30',
               '2025-12-31', '2026-01-01'],
              dtype='datetime64[ns]', length=366, freq='D')

In [6]:
class TimeRangeSplit():
    """
        https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.date_range.html
    """
    def __init__(self, 
                 start_date, 
                 end_date=None, 
                 freq='D', 
                 periods=None, 
                 tz=None, 
                 normalize=False, 
                 train_min_date=None,
                 filter_cold_users=True, 
                 filter_cold_items=True, 
                 filter_already_seen=True):
        
        self.start_date = start_date
        if end_date is None and periods is None:
            raise ValueError('Either "end_date" or "periods" must be non-zero, not both at the same time.')

        self.end_date = end_date
        self.freq = freq
        self.periods = periods
        self.tz = tz
        self.normalize = normalize
        self.train_min_date = pd.to_datetime(train_min_date, errors='raise')
        self.filter_cold_users = filter_cold_users
        self.filter_cold_items = filter_cold_items
        self.filter_already_seen = filter_already_seen

        self.date_range = pd.date_range(
            start=start_date, 
            end=end_date, 
            freq=freq, 
            periods=periods, 
            tz=tz, 
            normalize=normalize, 
            )

        self.max_n_splits = max(0, len(self.date_range) - 1)
        if self.max_n_splits == 0:
            raise ValueError('Provided parametrs set an empty date range.') 

    def split(self, 
              df, 
              user_column='user_id',
              item_column='item_id',
              datetime_column='date',
              fold_stats=False):
        df_datetime = df[datetime_column]
        if self.train_min_date is not None:
            train_min_mask = df_datetime >= self.train_min_date
        else:
            train_min_mask = df_datetime.notnull()

        date_range = self.date_range[(self.date_range >= df_datetime.min()) & 
                                     (self.date_range <= df_datetime.max())]

        for start, end in pairwise(date_range):
            fold_info = {
                'Start date': start,
                'End date': end
            }
            train_mask = train_min_mask & (df_datetime < start)
            train_idx = df.index[train_mask]
            if fold_stats:
                fold_info['Train'] = len(train_idx)

            test_mask = (df_datetime >= start) & (df_datetime < end)
            test_idx = df.index[test_mask]
            
            if self.filter_cold_users:
                new = np.setdiff1d(
                    df.loc[test_idx, user_column].unique(), 
                    df.loc[train_idx, user_column].unique())
                new_idx = df.index[test_mask & df[user_column].isin(new)]
                test_idx = np.setdiff1d(test_idx, new_idx)
                test_mask = df.index.isin(test_idx)
                if fold_stats:
                    fold_info['New users'] = len(new)
                    fold_info['New users interactions'] = len(new_idx)

            if self.filter_cold_items:
                new = np.setdiff1d(
                    df.loc[test_idx, item_column].unique(), 
                    df.loc[train_idx, item_column].unique())
                new_idx = df.index[test_mask & df[item_column].isin(new)]
                test_idx = np.setdiff1d(test_idx, new_idx)
                test_mask = df.index.isin(test_idx)
                if fold_stats:
                    fold_info['New items'] = len(new)
                    fold_info['New items interactions'] = len(new_idx)

            if self.filter_already_seen:
                user_item = [user_column, item_column]
                train_pairs = df.loc[train_idx, user_item].set_index(user_item).index
                test_pairs = df.loc[test_idx, user_item].set_index(user_item).index
                intersection = train_pairs.intersection(test_pairs)
                print(f'Already seen number: {len(intersection)}')
                test_idx = test_idx[~test_pairs.isin(intersection)]
                # test_mask = rd.df.index.isin(test_idx)
                if fold_stats:
                    fold_info['Known interactions'] = len(intersection)

            if fold_stats:
                fold_info['Test'] = len(test_idx)

            yield (train_idx, test_idx, fold_info)

    def get_n_splits(self, df, datetime_column='date'):
        df_datetime = df[datetime_column]
        if self.train_min_date is not None:
            df_datetime = df_datetime[df_datetime >= self.train_min_date]

        date_range = self.date_range[(self.date_range >= df_datetime.min()) & 
                                     (self.date_range <= df_datetime.max())]

        return max(0, len(date_range) - 1)

In [7]:
def get_coo_matrix(df, 
                   user_col='user_id', 
                   item_col='item_id', 
                   weight_col=None, 
                   users_mapping={}, 
                   items_mapping={}):
    
    if weight_col is None:
        weights = np.ones(len(df), dtype=np.float32)
    else:
        weights = df[weight_col].astype(np.float32)

    interaction_matrix = sp.coo_matrix((
        weights, 
        (
            df[user_col].map(users_mapping.get), 
            df[item_col].map(items_mapping.get)
        )
    ))
    return interaction_matrix

In [8]:
def generate_implicit_recs_mapper(
    model,
    train_matrix,
    top_N,
    user_mapping,
    item_inv_mapping,
    filter_already_liked_items
):
    def _recs_mapper(user):
        user_id = user_mapping[user]
        recs = model.recommend(user_id, 
                               train_matrix, 
                               N=top_N, 
                               filter_already_liked_items=filter_already_liked_items)
        return [item_inv_mapping[item] for item in recs[0]]
    return _recs_mapper

## Getting data

In [9]:
%ls

 ��� � ���ன�⢥ C ����� ���� Windows 11
 ��਩�� ����� ⮬�: DAD0-2403

 ����ন��� ����� c:\Users\user\Desktop\code\ods_recsys_competition\discovery_notebooks

31.01.2026  21:46    <DIR>          .
31.01.2026  21:55    <DIR>          ..
31.01.2026  19:49            65�002 RecSys notebook Baseline.ipynb
31.01.2026  12:24           660�956 RecSys notebook EDA.ipynb
01.02.2026  10:41             8�151 test_rectools.ipynb
               3 䠩���        734�109 ����
               2 �����  1�738�789�138�432 ���� ᢮�����


In [10]:
users_df = pd.read_csv('../data/users_processed.csv',)
items_df = pd.read_csv('../data/items_processed.csv',)
interactions_df = pd.read_csv('../data/interactions_processed.csv', parse_dates=['last_watch_dt'])

# Baseline - популярное

In [13]:
class PopularRecommender():
    def __init__(self, max_K=10, days=30, item_column='item_id', dt_column='date'):
        self.max_K = max_K
        self.days = days
        self.item_column = item_column
        self.dt_column = dt_column
        self.recommendations = []
        
    def fit(self, df, ):
        min_date = df[self.dt_column].max().normalize() - pd.DateOffset(days=self.days)
        self.recommendations = df.loc[df[self.dt_column] > min_date, self.item_column].value_counts().head(self.max_K).index.values
    
    def recommend(self, users=None, N=10):
        recs = self.recommendations[:N]
        if users is None:
            return recs
        else:
            return list(islice(cycle([recs]), len(users)))

### Пример на одном фолде

In [14]:
test = interactions_df[interactions_df['last_watch_dt'] == interactions_df['last_watch_dt'].max()]
train = interactions_df[interactions_df['last_watch_dt'] < interactions_df['last_watch_dt'].max()]

In [15]:
pop_model = PopularRecommender(days=7, dt_column='last_watch_dt')
pop_model.fit(train)

In [16]:
top10_recs = pop_model.recommend()
top10_recs

array([ 9728, 15297, 10440, 13865, 12360, 14488, 12192,   341,   512,
        4151], dtype=int64)

In [17]:
item_titles = pd.Series(items_df['title'].values, index=items_df['item_id']).to_dict()

In [18]:
list(map(item_titles.get, top10_recs))

['гнев человеческий',
 'клиника счастья',
 'хрустальный',
 'девятаев',
 'круэлла',
 'мастер меча',
 'фемида видит',
 'лето - это море',
 'рядовой чээрин',
 'секреты семейной жизни']

In [19]:
recs = pd.DataFrame({'user_id': test['user_id'].unique()})
top_N = 10
recs['item_id'] = pop_model.recommend(recs['user_id'], N=top_N)
recs.head()

,user_id,item_id
0,203219,"[9728, 15297, 10440, 13865, 12360, 14488, 1219..."
1,125519,"[9728, 15297, 10440, 13865, 12360, 14488, 1219..."
2,626036,"[9728, 15297, 10440, 13865, 12360, 14488, 1219..."
3,1029980,"[9728, 15297, 10440, 13865, 12360, 14488, 1219..."
4,830261,"[9728, 15297, 10440, 13865, 12360, 14488, 1219..."


In [20]:
recs = recs.explode('item_id')

In [21]:
recs['rank'] = recs.groupby('user_id').cumcount() + 1
recs.head(top_N + 2)

,user_id,item_id,rank
0,203219,9728,1
0,203219,15297,2
0,203219,10440,3
0,203219,13865,4
0,203219,12360,5
0,203219,14488,6
0,203219,12192,7
0,203219,341,8
0,203219,512,9
0,203219,4151,10


In [22]:
compute_metrics(train, test, recs, 10)

MAP@10        0.089383
Novelty@10    4.528709
dtype: float64

# Валидация на фолдах

Возьмем 3 последние недели из наших данных, и будем тестировать на них последовательно (1 test fold - 1 неделя).

Не забывайте про проблему холодного старта.

In [23]:
last_date = interactions_df['last_watch_dt'].max().normalize()
folds = 3
start_date = last_date - pd.Timedelta(days=folds*7)
start_date, last_date

(Timestamp('2021-08-01 00:00:00'), Timestamp('2021-08-22 00:00:00'))

In [24]:
cv = TimeRangeSplit(start_date=start_date, periods=folds+1, freq='W')

cv.max_n_splits, cv.get_n_splits(interactions_df, datetime_column='last_watch_dt')

(3, 3)

In [25]:
cv.date_range

DatetimeIndex(['2021-08-01', '2021-08-08', '2021-08-15', '2021-08-22'], dtype='datetime64[ns]', freq='W-SUN')

In [26]:
folds_with_stats = list(cv.split(
    interactions_df, 
    user_column='user_id',
    item_column='item_id',
    datetime_column='last_watch_dt',
    fold_stats=True
))

folds_info_with_stats = pd.DataFrame([info for _, _, info in folds_with_stats])

Already seen number: 0
Already seen number: 0
Already seen number: 0


In [27]:
folds_info_with_stats

,Start date,End date,Train,New users,New users interactions,New items,New items interactions,Known interactions,Test
0,2021-08-01,2021-08-08,4203885,53408,112764,174,7020,0,264039
1,2021-08-08,2021-08-15,4587708,54662,111580,152,9282,0,276699
2,2021-08-15,2021-08-22,4985269,56014,116629,114,5954,0,297228


# Популярное на фолдах

In [28]:
top_N = 10
last_n_days = 7

In [29]:
final_results = []
validation_results = pd.DataFrame()

for fold, (train_idx, test_idx, info) in enumerate(folds_with_stats):
    train = interactions_df.loc[train_idx]
    test = interactions_df.loc[test_idx]
        
    pop_model = PopularRecommender(days=last_n_days, dt_column='last_watch_dt')
    pop_model.fit(train)

    recs = pd.DataFrame({'user_id': test['user_id'].unique()})
    recs['item_id'] = pop_model.recommend(recs['user_id'], N=top_N)
    recs = recs.explode('item_id')
    recs['rank'] = recs.groupby('user_id').cumcount() + 1

    fold_result = pd.DataFrame([compute_metrics(train, test, recs, top_N)])
    fold_result["fold"] = fold
    validation_results = pd.concat([validation_results, fold_result])
validation_results

,MAP@10,Novelty@10,fold
0,0.081606,4.095690,0
0,0.075371,4.262035,1
0,0.070339,4.168475,2


In [30]:
validation_results.agg({'MAP@10':'mean', 'Novelty@10':'mean'})

MAP@10        0.075772
Novelty@10    4.175400
dtype: float64

# Соцдем популярное 

Посмотрим, имеет ли смысл предсказывать популярное в зависимости от соц.группы

In [50]:
train_idx, test_idx, info = folds_with_stats[0]
train = interactions_df.loc[train_idx]
test = interactions_df.loc[test_idx]
date_window_for_popular = train['last_watch_dt'].max() - pd.DateOffset(days=last_n_days)
train_slice = pd.merge(train[train['last_watch_dt'] >= date_window_for_popular], users_df, on='user_id', how='left')

Как мы помним из предыдущего ноутбука, у нас есть пользователи без фичей, поэтому для них надо определить заполнение 

In [51]:
train_slice.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct,age,income,sex,kids_flg
0,791466,8199,2021-07-27,713,9,age_18_24,income_20_40,F,False
1,81786,2616,2021-07-24,41422,90,age_35_44,income_20_40,F,True
2,161176,10440,2021-07-29,22,0,age_25_34,income_0_20,F,False
3,513902,3614,2021-07-24,1164,5,NaN,NaN,NaN,NaN
4,568405,15297,2021-07-30,15298,100,age_18_24,income_40_60,F,False


In [52]:
train_slice.fillna({'age':'age_unknown',
                    'sex':'sex_unknown',
                    'income': 'income_unknown',
                    'kids_flg': False
                   }, inplace=True)

C:\Users\user\AppData\Local\Temp\ipykernel_10476\21367217.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_slice.fillna({'age':'age_unknown',


Например, можно смотреть популярное в разрезе возраста, пола и наличия детей

In [53]:
train_slice.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct,age,income,sex,kids_flg
0,791466,8199,2021-07-27,713,9,age_18_24,income_20_40,F,False
1,81786,2616,2021-07-24,41422,90,age_35_44,income_20_40,F,True
2,161176,10440,2021-07-29,22,0,age_25_34,income_0_20,F,False
3,513902,3614,2021-07-24,1164,5,age_unknown,income_unknown,sex_unknown,False
4,568405,15297,2021-07-30,15298,100,age_18_24,income_40_60,F,False


In [54]:
soc_dem_recommendations = train_slice.groupby(
    ['age', 'sex', 'income', 'item_id']
).size().to_frame().reset_index()

In [55]:
soc_dem_recommendations

,age,sex,income,item_id,0
0,age_18_24,F,income_0_20,14,7
1,age_18_24,F,income_0_20,24,1
2,age_18_24,F,income_0_20,28,1
3,age_18_24,F,income_0_20,85,1
4,age_18_24,F,income_0_20,98,1
...,...,...,...,...,...
74358,age_unknown,sex_unknown,income_unknown,16499,32
74359,age_unknown,sex_unknown,income_unknown,16505,2
74360,age_unknown,sex_unknown,income_unknown,16506,1
74361,age_unknown,sex_unknown,income_unknown,16509,199


Теперь надо просто для каждого пользователя выбрать самое популярные top_n объектов в его группе

Можем проверить этот вариант на фолдах

In [60]:
validation_results = pd.DataFrame()

for train_idx, test_idx, info in folds_with_stats:
    train = interactions_df.loc[train_idx]
    test = interactions_df.loc[test_idx]
    date_window = train['last_watch_dt'].max() - pd.DateOffset(days=last_n_days)
    train_slice = pd.merge(train[train['last_watch_dt'] >= date_window], users_df, on='user_id', how='left')
    
    train_slice.fillna({
        'age':'age_unknown',
        'sex':'sex_unknown',
        'income': 'income_unknown',
        'kids_flg': False
    },inplace=True)
    
    soc_dem_recommendations = train_slice.groupby(
        ['age', 'sex', 'income', 'item_id']
    ).size().to_frame().reset_index()
    
    top_soc_dem = []

    for age in soc_dem_recommendations.age.unique():
        for income in soc_dem_recommendations.income.unique():
            for sex in soc_dem_recommendations.sex.unique():
                top_items = soc_dem_recommendations[
                (soc_dem_recommendations.age == age)
                & (soc_dem_recommendations.income == income)
                & (soc_dem_recommendations.sex == sex)].sort_values(0, ascending=False).head(10).item_id.values
                top_soc_dem.append([age, income, sex, top_items])

    top_soc_dem = pd.DataFrame(top_soc_dem, columns = ['age', 'income', 'sex', 'item_id'])
    
    recs = pd.DataFrame({'user_id': test['user_id'].unique()})
    recs = pd.merge(recs[['user_id']], users_df, on='user_id', how='left')
    recs.fillna({
        'age':'age_unknown',
        'sex':'sex_unknown',
        'income': 'income_unknown',
        'kids_flg': False
    }, inplace=True)
    
    recs = pd.merge(recs, top_soc_dem, on = ['age', 'sex', 'income'], how = 'left')
    recs = recs.drop(columns = ['age', 'sex', 'income'])
    
    recs = recs.explode('item_id')
    recs['rank'] = recs.groupby('user_id').cumcount() + 1
    fold_result = pd.DataFrame([compute_metrics(train, test, recs, top_N)])
    
    validation_results = pd.concat([validation_results, fold_result])

C:\Users\user\AppData\Local\Temp\ipykernel_10476\2631512229.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_slice.fillna({
C:\Users\user\AppData\Local\Temp\ipykernel_10476\2631512229.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  recs.fillna({
C:\Users\user\AppData\Local\Temp\ipykernel_10476\2631512229.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.n

In [61]:
validation_results.agg({'MAP@10':'mean', 'Novelty@10':'mean'})

MAP@10        0.080482
Novelty@10    4.237977
dtype: float64

В данном случае признаки, по которым вы строите популярное, подбираются, также, как и кол-во дней, которое вы берете для расчета популярного 

# Tfidf

In [105]:
users_inv_mapping = dict(enumerate(interactions_df['user_id'].unique()))
users_mapping = {v: k for k, v in users_inv_mapping.items()}

items_inv_mapping = dict(enumerate(interactions_df['item_id'].unique()))
items_mapping = {v: k for k, v in items_inv_mapping.items()}

In [111]:
validation_results = pd.DataFrame()

for train_idx, test_idx, info in folds_with_stats:
    train = interactions_df.loc[train_idx]

    date_window = train['last_watch_dt'].max() - pd.DateOffset(days=60)
    train = train[train['last_watch_dt'] >= date_window]

    test = interactions_df.loc[test_idx]

    train_mat = get_coo_matrix(
        train,
        users_mapping=users_mapping,
        items_mapping=items_mapping,
    ).tocsr()

    model = TFIDFRecommender(K=5000)
    model.fit(train_mat, show_progress=False) 

    mapper = generate_implicit_recs_mapper( 
        model,
        train_mat,
        top_N,
        users_mapping,
        items_inv_mapping,
        filter_already_liked_items=True
    )

    recs = pd.DataFrame({'user_id': test['user_id'].unique()})
    recs['item_id'] = recs['user_id'].map(mapper)
    recs = recs.explode('item_id')
    recs['rank'] = recs.groupby('user_id').cumcount() + 1
    fold_result = pd.DataFrame([compute_metrics(train, test, recs, top_N)])

    validation_results = pd.concat([validation_results, fold_result])
validation_results

c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\ods-recsys-competition-kzVvw0Ap-py3.10\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.010680675506591797 seconds
  warnings.warn(
c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\ods-recsys-competition-kzVvw0Ap-py3.10\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.01300048828125 seconds
  warnings.warn(
c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\ods-recsys-competition-kzVvw0Ap-py3.10\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.013999462127685547 seconds
  warnings.warn(


,MAP@10,Novelty@10
0,0.025621,6.459808
0,0.027840,5.766509
0,0.019952,6.225828


In [112]:
max(list(items_inv_mapping.keys()))

15705

In [119]:
user_id = 0
recs = model.recommend(user_id, 
                    train_mat, 
                    N=top_N, 
                    filter_already_liked_items=True)
recs

(array([  84,   11,  986, 1440,   16,  363,  962,  481,  516,  423]),
 array([1.12425048, 1.11421743, 1.09856754, 1.0428479 , 1.02801769,
        0.98565158, 0.98035444, 0.9769162 , 0.97356134, 0.9501793 ]))

In [87]:
max(list(items_inv_mapping.keys()))

15705

In [121]:
validation_results.agg({'MAP@10':'mean', 'Novelty@10':'mean',})

MAP@10        0.024471
Novelty@10    6.150715
dtype: float64

Просто использовать код выше для submission не получится из-за холодных пользователей. Придется придумать, как их обработать.

# Пример submission

In [124]:
submission = pd.read_csv('../data/sample_submission.csv')

In [125]:
train = interactions_df
test = submission

pop_model = PopularRecommender(days=last_n_days, dt_column='last_watch_dt')
pop_model.fit(train)

recs = pd.DataFrame({'user_id': test['user_id'].unique()})
recs['item_id'] = pop_model.recommend(recs['user_id'], N=top_N)
recs = recs.explode('item_id')
recs['rank'] = recs.groupby('user_id').cumcount() + 1
recs = recs.groupby('user_id').agg({'item_id': list}).reset_index()

In [126]:
recs.head()

,user_id,item_id
0,3,"[9728, 15297, 10440, 14488, 13865, 12192, 341,..."
1,11,"[9728, 15297, 10440, 14488, 13865, 12192, 341,..."
2,29,"[9728, 15297, 10440, 14488, 13865, 12192, 341,..."
3,30,"[9728, 15297, 10440, 14488, 13865, 12192, 341,..."
4,33,"[9728, 15297, 10440, 14488, 13865, 12192, 341,..."


In [127]:
recs.to_csv('sample_submission.csv', index=False)

### Тест RecTools

In [45]:
import pandas as pd
from implicit.nearest_neighbours import TFIDFRecommender, BM25Recommender

from rectools import Columns
from rectools.dataset import Dataset
from rectools.models import ImplicitItemKNNWrapperModel, RandomModel, PopularModel

In [12]:
interactions_df = interactions_df.rename(columns={"watched_pct": "weight", "last_watch_dt": "datetime"})
interactions_df

,user_id,item_id,datetime,total_dur,weight
0,176549,9506,2021-05-11,4250,72
1,699317,1659,2021-05-29,8317,100
2,656683,7107,2021-05-09,10,0
3,864613,7638,2021-07-05,14483,100
4,964868,9506,2021-04-30,6725,100
...,...,...,...,...,...
5476246,648596,12225,2021-08-13,76,0
5476247,546862,9673,2021-04-13,2308,49
5476248,697262,15297,2021-08-20,18307,63
5476249,384202,16197,2021-04-19,6203,100


In [14]:
dataset = Dataset.construct(interactions_df)
dataset

Dataset(user_id_map=IdMap(external_ids=array([176549, 699317, 656683, ..., 805174, 648596, 697262], dtype=int64)), item_id_map=IdMap(external_ids=array([ 9506,  1659,  7107, ..., 10064, 13019, 10542], dtype=int64)), interactions=Interactions(df=         user_id  item_id  weight   datetime
0              0        0    72.0 2021-05-11
1              1        1   100.0 2021-05-29
2              2        2     0.0 2021-05-09
3              3        3   100.0 2021-07-05
4              4        0   100.0 2021-04-30
...          ...      ...     ...        ...
5476246   962177      208     0.0 2021-08-13
5476247   224686     2690    49.0 2021-04-13
5476248   962178       21    63.0 2021-08-20
5476249     7934     1725   100.0 2021-04-19
5476250   631989      157    45.0 2021-08-15

[5476251 rows x 4 columns]), user_features=None, item_features=None)

In [15]:
model = ImplicitItemKNNWrapperModel(TFIDFRecommender(K=10))
model.fit(dataset)

# Make recommendations
recos = model.recommend(
    users=interactions_df[Columns.User].unique(),
    dataset=dataset,
    k=10,
    filter_viewed=True,
)

c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\ods-recsys-competition-kzVvw0Ap-py3.10\lib\site-packages\implicit\nearest_neighbours.py:233: RuntimeWarning: invalid value encountered in divide
  X.data = X.data / sqrt(bincount(X.row, X.data**2))[X.row]


In [18]:
from rectools.model_selection.time_split import TimeRangeSplitter
time_split = TimeRangeSplitter(test_size="7D",
                               n_splits=3,
                               filter_cold_users=False,
                               filter_cold_items=False,
                               filter_already_seen=False)
time_split

In [20]:
from rectools.dataset.interactions import Interactions

interactions = Interactions(interactions_df)
interactions

Interactions(df=         user_id  item_id   datetime  total_dur  weight
0         176549     9506 2021-05-11       4250    72.0
1         699317     1659 2021-05-29       8317   100.0
2         656683     7107 2021-05-09         10     0.0
3         864613     7638 2021-07-05      14483   100.0
4         964868     9506 2021-04-30       6725   100.0
...          ...      ...        ...        ...     ...
5476246   648596    12225 2021-08-13         76     0.0
5476247   546862     9673 2021-04-13       2308    49.0
5476248   697262    15297 2021-08-20      18307    63.0
5476249   384202    16197 2021-04-19       6203   100.0
5476250   319709     4436 2021-08-15       3921    45.0

[5476251 rows x 5 columns])

In [49]:
from rectools.model_selection.cross_validate import cross_validate
from rectools.metrics.ranking import MAP
from rectools.metrics.classification import HitRate, Precision, Recall
from rectools.metrics.novelty import MeanInvUserFreq


In [50]:
models = {
    "random": RandomModel(random_state=42),
    "popular": PopularModel(),
    "most_rated": PopularModel(popularity="sum_weight"),
    "tfidf_k=10": ImplicitItemKNNWrapperModel(model=TFIDFRecommender(K=10)),
    "bm25_k=10_k1=0.05_b=0.1": ImplicitItemKNNWrapperModel(model=BM25Recommender(K=5, K1=0.05, B=0.1)),
}
metrics = {
    "MAP@10": MAP(k=10),
    "prec@10": Precision(k=10),
    "recall@10": Recall(k=10),
    "novelty@10": MeanInvUserFreq(k=10),    
    "hit10": HitRate(k=10),
}

K_RECS = 10

In [ ]:
cv_results = cross_validate(
    dataset=dataset,
    splitter=time_split,
    models=models,
    metrics=metrics,
    k=K_RECS,
    filter_viewed=True,
)

c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\ods-recsys-competition-kzVvw0Ap-py3.10\lib\site-packages\implicit\nearest_neighbours.py:233: RuntimeWarning: invalid value encountered in divide
  X.data = X.data / sqrt(bincount(X.row, X.data**2))[X.row]
c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\ods-recsys-competition-kzVvw0Ap-py3.10\lib\site-packages\rectools\models\base.py:728: UserWarning: 
                Model `<class 'rectools.models.implicit_knn.ImplicitItemKNNWrapperModel'>` doesn't support recommendations for cold users,
                but some of given users are cold: they are not in the `dataset.user_id_map`
            
  warnings.warn(explanation)
c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\ods-recsys-competition-kzVvw0Ap-py3.10\lib\site-packages\rectools\models\base.py:728: UserWarning: 
                Model `<class 'rectools.models.implicit_knn.ImplicitItemKNNWrapperModel'>` doesn't support recommendations for cold users,
                bu

In [ ]:
pd.DataFrame(cv_results['metrics'])

In [ ]:
for train_idx, test_idx, info in time_split.split(interactions):
    model = ImplicitItemKNNWrapperModel(TFIDFRecommender(K=10))
    model.fit(dataset)

    # Make recommendations
    recos = model.recommend(
        users=interactions_df[Columns.User].unique(),
        dataset=dataset,
        k=10,
        filter_viewed=True,
    )

383149
402653
424436
